 # Block ciphers

 ## Feistel networks

 ![Feistel networks](./feistel_network.png)

 ## Confusion and Diffusion

 Claude Shannon 1945:
 - confusion: every bit of the ciphertext depends on some part of the key
 - diffusion: if we change one bit of the plaintext, then half of the bits of the ciphertext should change. Similarly, if we change one bit of the ciphertext, then half of the bits of the plaintext should change.

 ## bytestring in Python

 Byte sequences, on ASCII characters. [Documentation](https://docs.python.org/3/library/stdtypes.html#binary-sequence-types-bytes-bytearray-memoryview)

 ![ASCII table](./ascii_table.png)

In [0]:
b"cryptography is fun"


In [0]:
b"cryptography is fun".hex()


In [0]:
b"\x20a\x00\xff".hex()


 If we iterate over the bytestring, we get the decimal representation of each character:

In [0]:
for i in b"crypto\x01":
    print(i)


 Indexing also outputs the decimal value of the byte:

In [0]:
v = b"help\x04\xff"
v[-1], v[0]


 Hexadecimal numbers:

In [0]:
for i in range(20):
    print(f"{i:>2}: {i:08b} {i:02x}")


In [0]:
0xff


 You can convert an integer to a hexadecimal number using the `hex()` function:

In [0]:
hex(15)


In [0]:
def xor_strings(a, b):
    m = b""
    for i, j in zip(a, b):
        m += bytes.fromhex(f"{i ^^ j:02x}")
    return m


In [0]:
xor_strings(b"hello", b"there").hex()


 ## PKCS #5 and PKCS #7 padding

 PKCS: Public Key Cryptography Standards

 Goal: the length of the plaintext should be a multiple of the block size.

 The value of the padding is the number of bytes added to the plaintext.

 **Example**: Let's say the block size is 8 bytes, but the message is 12 bytes. Then the padding:

 `DD DD DD DD DD DD DD DD | DD DD DD DD 04 04 04 04 |`

 If the message is block size long, then we add a new block

In [0]:
def padding(block_size: int, msg: bytes) -> bytes:
    pad_length = block_size - len(msg) % block_size

    return msg + bytes([pad_length]) * pad_length


In [0]:
padding(8, b"cryptography")


In [0]:
padding(12, b"cryptography")


 Other padding methods:
 - bit padding: one `1` followed by as many `0`s as needed
 - ISO/IEC 7816-4: `80` followed by as many `0`s as needed
 - Zero padding: as many `0`s as needed

 ## Data Encryption Standard

 1977, [NIST standard (pdf)](https://csrc.nist.gov/CSRC/media/Publications/fips/46/3/archive/1999-10-25/documents/fips46-3.pdf)

 Properties:
 - 64 (56) bit long key
 - 64 bit long block
 - Feistel network with 16 rounds

 ![DES](./des.png)

In [0]:
from operator import xor
import secrets
from typing import List, Optional

from permutations import *


In [0]:
class BadPaddingError(Exception):
    pass


In [0]:
def xor_blocks(a: List[int], b: List[int]) -> List[int]:
    return [xor(i, j) for i, j in zip(a, b)]


def check_padding(s: bytes) -> bool:
    return all(s[-1] == i for i in s[-s[-1]:])


def xor_strings(a, b):
    m = b""
    for i, j in zip(a, b):
        m += bytes.fromhex(f'{xor(i, j):02x}')
    return m


In [0]:
class DES:
    def __init__(self, key: bytes, block_size: int) -> None:
        if len(key) != 8:
            raise ValueError(f'The key must be 8 bytes long')
        self.key = key
        self.block_size = block_size
        self.round_keys = self._key_schedule()

    def encrypt(self, plaintext: bytes, apply_pad: bool) -> bytes:
        raise NotImplementedError

    def decrypt(self, ciphertext: bytes, apply_pad: bool) -> bytes:
        raise NotImplementedError

    def _encryption(self, block: bytes, encrypt: bool) -> bytes:
        block = int.from_bytes(block, byteorder='big')
        block = list(map(int, bin(block)[2:].zfill(64)))
        block = [block[i] for i in IP]
        left, right = block[:32], block[32:]
        if encrypt:
            round_keys = self.round_keys
        else:
            round_keys = self.round_keys[::-1]
        for n in range(16):
            round_key = round_keys[n]
            left, right = right, xor_blocks(left, self._cipher(right, round_key))
        block = right + left
        block = [block[i] for i in IP_inv]
        block = int('0b' + ''.join(str(b) for b in block), base=2).to_bytes(8, byteorder='big')
        return block

    def _cipher(self, r: List[int], k: List[int]) -> List[int]:
        subst = [S1, S2, S3, S4, S5, S6, S7, S8]
        r = [r[i] for i in E]
        rk = xor_blocks(r, k)
        res, j = [], 0
        for i in range(0, 48 - 6 + 1, 6):
            block = rk[i:i+6]
            row, col = int(f'0b{block[0]}{block[-1]}', base=2), int(f'0b{"".join(str(b) for b in block[1:-1])}', base=2)
            res += [int(b) for b in bin(subst[j][row][col])[2:].zfill(4)]
            j += 1
        res = [res[i] for i in P]
        return res

    def _key_schedule(self) -> List[List[int]]:
        iterations = [1, 1, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 1]
        res = []
        key = int.from_bytes(self.key, byteorder='big')
        key = list(map(int, f"{key:064b}"))
        round_key = [key[i] for i in PC1]
        c, d = round_key[:28], round_key[28:]
        for n in range(16):
            left_shifts = iterations[n]
            c = c[left_shifts:] + c[:left_shifts]
            d = d[left_shifts:] + d[:left_shifts]
            round_key = c + d
            res.append([round_key[i] for i in PC2])
        return res


 ## Modes of operation

 ### Electronic Code Book (ECB)

 ![ECB mode](./mode_ecb.png)

 The encrypted text: $c = F(k, m_1)\,||\,F(k, m_2)\,||\,\cdots\,||\,F(k, m_\ell))$

 Decryption: $p = F^{-1}(k, c_1)\,||\,F^{-1}(k, c_2)\,||\,\cdots\,||\,F^{-1}(k, c_{\ell})$

 ### Cipher Block Chaining (CBC)

 ![CBC mode](./mode_cbc.png)

 The encrypted text consists of several parts:
 - $c_0 = IV$
 - $c_i = F(k, c_{i-1} \oplus m_{i})$
 - and $c = c_0\,||\,c_1 ||\,\cdots\,||c_{\ell}$

 Decryption: $m_i = F^{-1}(k,c_i) \oplus c_{i-1}$ for every $1 \leq i \leq \ell$ block

 **Task**: Encrypt the ELTE coat of arms with AES encryption in ECB and CBC modes!

 **Solution**:
 1. ECB mode:
 ```bash
 head -n 4 elte_cimer.ppm > header.txt
 tail -n +5 elte_cimer.ppm > body.bin
 openssl enc -aes-128-ecb -nosalt -pass pass:"VeryLongPassword" -in body.bin -out body.ecb.bin
 cat header.txt body.ecb.bin > elte_cimer_ecb.ppm
 ```
 2. CBC mode (the IV is randomly generated):
 ```bash
 head -n 4 elte_cimer.ppm > header.txt
 tail -n +5 elte_cimer.ppm > body.bin
 openssl enc -aes-128-cbc -nosalt -pass pass:"VeryLongPassword" -in body.bin -out body.cbc.bin
 cat header.txt body.cbc.bin > elte_cimer_cbc.ppm
 ```

In [0]:
class DES_ECB(DES):
    def __init__(self, key: bytes, block_size: int = 8) -> None:
        super().__init__(key, block_size)

    def encrypt(self, plaintext: bytes, apply_pad: bool = True) -> bytes:
        res = b""
        if apply_pad:
            plaintext = padding(self.block_size, plaintext)
        for b in range(0, len(plaintext), self.block_size):
            pt_block = plaintext[b:b+self.block_size]
            res += self._encryption(pt_block, encrypt=True)
        return res

    def decrypt(self, ciphertext: bytes, apply_pad: bool = True) -> bytes:
        res = b""
        for b in range(0, len(ciphertext), self.block_size):
            pt_block = ciphertext[b:b+self.block_size]
            res += self._encryption(pt_block, encrypt=False)
        if apply_pad:
            if check_padding(res):
                return res[:-res[-1]]
            raise BadPaddingError('Bad padding')
        return res


In [0]:
class DES_CBC(DES):
    def __init__(self, key: bytes, block_size: int = 8, iv: Optional[bytes] = None) -> None:
        super().__init__(key, block_size)
        if iv is None:
            iv = secrets.token_bytes(block_size)
        self.iv = iv
        # equivalent to: self.iv = iv or secrets.token_bytes(block_size)

    def encrypt(self, plaintext: bytes, apply_pad: bool = True) -> bytes:
        res = self.iv
        if apply_pad:
            plaintext = padding(self.block_size, plaintext)
        for b in range(0, len(plaintext), self.block_size):
            pt_block = plaintext[b:b+self.block_size]
            x = xor_strings(res[-self.block_size:], pt_block)
            res += self._encryption(x, encrypt=True)
        return res

    def decrypt(self, ciphertext: bytes, apply_pad: bool = True) -> bytes:
        res = b""
        for b in range(self.block_size, len(ciphertext), self.block_size):
            ct_block = ciphertext[b:b+self.block_size]
            t = self._encryption(ct_block, encrypt=False)
            res += xor_strings(t, ciphertext[b-self.block_size:b])
        if apply_pad:
            if check_padding(res):
                return res[:-res[-1]]
            raise BadPaddingError('Bad padding')
        return res


In [0]:
des = DES_ECB(secrets.token_bytes(8))
des.decrypt(des.encrypt(b'cryptography'))


 ### Weak and semi-weak keys

 Let $w_1,w_2$ be keys and $m$ a message.
 - if $\text{DES}(w_1, \text{DES}(w_1, m)) = m,$ then the $w_1$ key is weak.
 - if $\text{DES}(w_1, \text{DES}(w_2, m)) = m,$ then the $w_1, w_2$ key pair is a semi-weak key pair.

 Weak keys:
 - `0101010101010101`
 - `fefefefefefefefe`
 - `1f1f1f1f0e0e0e0e`
 - `e0e0e0e0f1f1f1f1`

 Semi-weak key pairs:
 - `01fe01fe01fe01fe` and `fe01fe01fe01fe01`
 - `1fe01fe00ef10ef1` and `e01fe01ff10ef10e`
 - `01e001e001f001f1` and `e001e001f101f101`
 - `1ffe1ffe0efe0efe` and `fe1ffe1ffe0efe0e`
 - `011f011f010e010e` and `1f011f010e010e01`
 - `e0fee0fef1fef1fe` and `fee0fee0fef1fef1`

In [0]:
for k in ['0101010101010101', 'fefefefefefefefe', '1f1f1f1f0e0e0e0e', 'e0e0e0e0f1f1f1f1']:
    des = DES_ECB(bytes.fromhex(k))
    print(des.encrypt(des.encrypt(b'animator', apply_pad=False), apply_pad=False))


In [0]:
for k1, k2 in [('01fe01fe01fe01fe', 'fe01fe01fe01fe01'),
               ('1fe01fe00ef10ef1', 'e01fe01ff10ef10e'),
               ('01e001e001f001f1', 'e001e001f101f101'),
               ('1ffe1ffe0efe0efe', 'fe1ffe1ffe0efe0e'),
               ('011f011f010e010e', '1f011f010e010e01'),
               ('e0fee0fef1fef1fe', 'fee0fee0fef1fef1')]:
    des1 = DES_ECB(bytes.fromhex(k2))
    des2 = DES_ECB(bytes.fromhex(k1))
    print(des2.encrypt(des1.encrypt(b'animator', apply_pad=False), apply_pad=False))


 ## Padding-oracle attack on CBC mode

 **Task**: Break the CBC mode of operation!

 We know the length of a block (this is included in the standards)

 The encrypted text consists of several parts:
 - $c_0 = IV$
 - $c_i = F(k, c_{i-1} \oplus m_{i})$
 - and $c = c_0\,||\,c_1 ||\,\cdots\,||c_{\ell}$

 Decryption: $m_i = F^{-1}(k,c_i) \oplus c_{i-1}$ for every $1 \leq i \leq \ell$ block

 - padding check during decryption: the last $n$ bytes contain the value $n$
 - idea: changing certain bytes in the ciphertext causes a predictable change in the decryption

In [0]:
plaintext = b"cryptography"

pt = padding(8, plaintext)
pt


In [0]:
des = DES_CBC(b"mysecret")
ct = des.encrypt(plaintext)
ct, len(ct)


 **Example**: Let the plaintext be the above text, the ciphertext is then $IV||c_1||c_2$.

 Steps of the break:
 1. Find the padding size (i.e., the length of the plaintext):
     - Modify the first byte of $c_1$ and decrypt the modified ciphertext
         - Then $m_2' = F^{-1}(k, c_2) \oplus c_1'$, i.e., only the first byte of $m_2$ will change
     - If we don't get an error, modify the next one
     - Where we first get an error, the padding is bad
         - From this we know the padding (and plaintext) length

In [0]:
for i in range(8):
    ct_ = ct[:8+i] + b"\x00" + ct[8+i+1:]
    print(f"{i:>2d}: {ct_}")
    try:
        pt_ = des.decrypt(ct_)
    except BadPaddingError:
        print(f"{i} is bad")
        b = i
        break


 2. Finding the last byte of the message
     - We know the padding length, $b$. Let's denote the last byte of the plaintext with $B$
     - We know that $m_2$ ends with $B\,0xb \cdots 0xb$ ($0xb$ is repeated exactly $b$ times)
         - In our example: `B\x04\x04\x04\x04`
     - Let $\Delta_i = 0x00 \cdots 0x00\;0xi\;\underbrace{0x(b+1) \cdots 0x(b+1)}_\text{b times} \oplus 0x00 \cdots 0x00\;0x00\;\underbrace{0xb \cdots 0xb}_\text{b times}$ for every $0 \leq i < 2^8$. The last $b+1$ bytes contain the value of $i$ and $(b+1) \oplus b$ in hexadecimal form
         - For example: $\Delta_{137} = $ `\x00\x00\x00\x89\x05\x05\x05\x05 ^ \x00\x00\x00\x00\x04\x04\x04\x04 = \x00\x00\x00\x89\x01\x01\x01\x01`
     - Decrypt the $IV|| c_1 \oplus \Delta_i || c_2$ ciphertext
         - The last $b+1$ bytes are in the form $0x(B \oplus i)\;\underbrace{0x(b+1) \cdots 0x(b+1)}_\text{b times}$
         - The decryption will be bad until $0x(B \oplus i) = 0x(b+1)$ is satisfied
         - The value of the last byte is $B = 0x(b+1) \oplus i$

In [0]:
prefix = b"\x00" * (8 - (b + 1))
for i in range(256):
    i_byte = bytes.fromhex(f"{i:02x}")
    l = prefix + i_byte + bytes.fromhex(f"{b+1:02x}") * b
    r = prefix + b"\x00" + bytes.fromhex(f"{b:02x}") * b
    delta_i = xor_strings(l, r)
    ct_ = ct[:8] + xor_strings(ct[8:16], delta_i) + ct[16:]
    try:
        des.decrypt(ct_)
        print(xor_strings(bytes.fromhex(f"{b+1:02x}"), bytes.fromhex(f"{i:02x}")))
        break
    except BadPaddingError:
        continue


 ## Meet-in-the-Middle attack

 **Question**: Given $k_1, k_2$ keys for some symmetric encryption. Does it provide extra security if we encrypt *twice*, i.e., $c = \texttt{Enc}(k_2, \texttt{Enc}(k_1, \text{plaintext}))$ with the following encryptions:
 - shift encryption
 - affine encryption: $(a, b) \in \mathbb{Z}_{26}^* \times \mathbb{Z}_{26}^*$
     - $\mathtt{Enc}: c \equiv ap + b \pmod{26}$,
     - $\mathtt{Dec}: p \equiv a^{-1}(c - b) \pmod{26}$
     - DES

 **Solution**: The shift and affine encryption (and all the classical ones) do not provide extra security. The $k = k_1 + k_2 \mod{26}$ encrypted key results in the same ciphertext. In the case of DES it seems good because we have a twice 56-bit long key (112-bit), and looking through $2^{112}$ possible keys still takes a long time. But unfortunately, it's not good because of the Meet-in-the-Middle attack.

 The method of the Meet-in-the-Middle attack:

 \begin{align*}
     c &= \mathtt{Enc}(k_2, \mathtt{Enc}(k_1, \text{plaintext})) \\
     \mathtt{Dec}(k_2, c) &= \mathtt{Dec}(k_2, \mathtt{Enc}(k_2, \mathtt{Enc}[k_1, \text{plaintext}])) \\
     \mathtt{Dec}(k_2, c) &= \mathtt{Enc}(k_1, \text{plaintext})
 \end{align*}

 Let's assume that Eve knows the $m$ message and the $c$ encrypted message derived from it (known plaintext attack). Goal: finding $k_1, k_2$.

 To break it, we need $\mathcal{O}\left( (n + \ell)\cdot 2^n \right)$ memory and $\mathcal{O}(n \cdot 2^n)$ time.

 Steps:
 1. For every $k_1$ key, calculate $z = \mathtt{Enc}(k_1, m)$ and store the $(z, k_1)$ pair in an $L$ list.
 2. For every $k_2$ key, calculate $z = \mathtt{Dec}(k_2, c)$ and store the $(z, k_2)$ pair in an $L'$ list.
 3. The $(z_1, k_1) \in L$ and $(z_2, k_2) \in L'$ elements match if $z_1 = z_2$. For every such pair, add the $(k_1, k_2)$ pair to an $S$ set.
     - This $S$ set only contains those $(k_1, k_2)$ pairs for which $\mathtt{Enc}(k_1, m) = \mathtt{Dec}(k_2, c)$
     - This $S$ set will contain the unknown key pair
 4. If we can do this for several $(m,c)$ pairs, the size of $S$ can be significantly reduced if we take their intersection.

In [0]:
import random
import secrets

des1 = DES_ECB(b"animator")
des2 = DES_ECB(b"reinvent")
plaintext = b"simplest"
ciphertext = des2.encrypt(des1.encrypt(plaintext, False), False)

words = []
with open("/usr/share/dict/words") as f:
    for word in f:
        if any(i in word.strip() for i in ["'", "ê", "é", "ü", "è", "â", "û", "ç", "ñ"]):
            continue
        if len(word) == 9:
            words.append(word.strip())

subwords = random.sample(words, k=8) + ["animator", "reinvent"]
random.shuffle(subwords)

outfile1 = open("meet-in-the-middle_1.txt", "w")
outfile2 = open("meet-in-the-middle_2.txt", "w")
for w in subwords:
    k = w.encode()
    des = DES_ECB(k)

    z = des.encrypt(plaintext, False)
    outfile1.write(f"{z} {k}\n")

    z = des.decrypt(ciphertext, False)
    outfile2.write(f"{z} {k}\n")

outfile2.close()
outfile1.close()


In [0]:
!head meet-in-the-middle_1.txt


In [0]:
!head meet-in-the-middle_2.txt


In [0]:
mitm1 = open("meet-in-the-middle_1.txt", "r")
mitm2 = open("meet-in-the-middle_2.txt", "r")

s = []

for items1 in mitm1:
    z1, k1 = items1.strip().rsplit(" ", maxsplit=1)
    for items2 in mitm2:
        z2, k2 = items2.strip().rsplit(" ", maxsplit=1)
        # print(z1, z2)
        if z1 == z2:
            s.append((k1, k2))
    mitm2.seek(0)

mitm2.close()
mitm1.close()

for i in s:
    print(i)


 And if we encrypt *three times*? We have two options:
 1. Choose $k_1, k_2, k_3$ keys and let $c = \mathtt{Enc}(k_3, \mathtt{Dec}(k_2, \mathtt{Enc}(k_1, x)))$
     - The key length is $3n$
     - Meet-in-the-Middle attack can be applied (it requires $\mathcal{O}(2^{2n})$ time)
 2. Choose $k_1, k_2$ keys and let $c = \mathtt{Enc}(k_1, \mathtt{Dec}(k_2, \mathtt{Enc}(k_1, x)))$
     - The key length is $2n$
     - No known attack better than $\mathcal{O}(2^{2n})$ is known

 3DES: standardized in 1999, but slow, the 2nd version uses too small a key